# Практика 3. Полная вероятность

## Перекладывание шаров между урнами

В первой урне находятся 15 белых и 5 чёрных шаров, во второй — 6 белых и 1 чёрный. Из первой урны случайно выбирают 3 шара без возвращения и перекладывают во вторую. Затем из второй урны извлекают один шар.

Найдём вероятность того, что извлечённый шар окажется белым, двумя способами: по формуле полной вероятности и с помощью моделирования.

Обозначим через $H_k$ событие «среди трёх переложенных шаров ровно $k$ белых», где $k=0,1,2,3$. Тогда

$$P(H_k)=\frac{C_{15}^{k}C_{5}^{3-k}}{C_{20}^{3}}.$$

После события $H_k$ во второй урне будет $6+k$ белых шаров из 10. По формуле полной вероятности:

$$P(A)=\sum_{k=0}^{3}P(H_k)P(A\mid H_k)
=\sum_{k=0}^{3}\frac{C_{15}^{k}C_{5}^{3-k}}{C_{20}^{3}}\cdot\frac{6+k}{10}.$$

In [ ]:
from math import comb


def probability_of_white_ball():
    total_combinations = comb(20, 3)
    return sum(
        comb(15, k) * comb(5, 3 - k) / total_combinations * (6 + k) / 10
        for k in range(4)
    )


p_theory = probability_of_white_ball()
print(f"Теоретическая вероятность: {p_theory:.6f}")

## Моделирование

В каждом опыте число белых шаров среди трёх переложенных имеет гипергеометрическое распределение. После переноса моделируем извлечение шара из второй урны и отслеживаем накопленную частоту.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(42)
n_trials = 100_000

moved_white = rng.hypergeometric(ngood=15, nbad=5, nsample=3, size=n_trials)
white_draws = rng.random(n_trials) < (6 + moved_white) / 10
cumulative_frequency = np.cumsum(white_draws) / np.arange(1, n_trials + 1)

p_experiment = white_draws.mean()
print(f"Частотная вероятность: {p_experiment:.6f}")
print(f"Абсолютное отклонение: {abs(p_experiment - p_theory):.6f}")

fig, ax = plt.subplots(figsize=(10, 5))
steps = np.arange(99, n_trials, 100)
ax.plot(steps + 1, cumulative_frequency[steps], label="Частота белого шара", color="teal")
ax.axhline(p_theory, color="darkorange", linestyle="--", label=f"Теория: {p_theory:.3f}")
ax.set(title="Сходимость частоты к теоретической вероятности", xlabel="Число опытов", ylabel="Вероятность")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

## Вывод

Классический расчёт даёт вероятность $0{,}825$. При увеличении числа опытов частота извлечения белого шара стабилизируется около этого значения, что согласуется с законом больших чисел.